In [1]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    set_seed
)
from datasets import Dataset as ds
import random
from peft import PeftModel
import pandas as pd
from tqdm.auto import tqdm

# Configurar semillas para facilitar la reproducibilidad de los resultados
seed = 44
torch.manual_seed(seed)
random.seed(seed)
set_seed(seed)

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device


'mps'

In [2]:
output_dir = "../modelos/gemma3_1b_it"

In [3]:
version_modelo = '../modelos/gemma3_1b_it/adaptador'

In [ ]:
from huggingface_hub import login
TOKEN = ''
login(token=TOKEN)

In [5]:
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-1b-it",device_map='auto', torch_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it", trust_remote_code=True)

model = PeftModel.from_pretrained(model, version_modelo)
model.config.pad_token_id = tokenizer.pad_token_id

'NoneType' object has no attribute 'cadam32bit_grad_fp32'


/Users/carlosivan/miniconda3/envs/pytorch/lib/python3.11/site-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['qalora_group_size', 'target_parameters', 'use_qalora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
/Users/carlosivan/miniconda3/envs/pytorch/lib/python3.11/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


In [6]:
model = model.merge_and_unload()

In [7]:
model.save_pretrained(output_dir, safe_serialization=True, max_shard_size="5GB")
tokenizer.save_pretrained(output_dir)

('../modelos/gemma3_1b_it/tokenizer_config.json',
 '../modelos/gemma3_1b_it/special_tokens_map.json',
 '../modelos/gemma3_1b_it/tokenizer.model',
 '../modelos/gemma3_1b_it/added_tokens.json',
 '../modelos/gemma3_1b_it/tokenizer.json')